# 05 · Fine-tune the four-class YOLOv8n detector
**Paper:** Step 4 · Table I

| Setting | Value |
|---|---|
| Base model | YOLOv8n (COCO-pretrained) |
| Input size | 512 × 512 |
| Batch size | 16 |
| Max epochs | 30 (early-stopping patience 10) |
| Classes | all four, trained jointly |
| Seed | 17 |

**Output** → `outputs/runs/<run_name>/`, best weights copied to `weights/requilt_yolov8n.pt`

In [ ]:
# --- setup: make the `requilt` package importable and load the shared config ---
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root (not needed after `pip install -e .`)

from requilt import load_config

cfg = load_config()      # configs/config.yaml
P = cfg.paths            # all paths, already resolved to absolute Paths

In [ ]:
import shutil

import pandas as pd
from IPython.display import Image as ShowImage
from ultralytics import YOLO

T = cfg.train
DATA_YAML = P.output_dir / "yolo_dataset" / "data.yaml"
RUNS_DIR = P.output_dir / "runs"

## 1 · Train
Training runs for at most 30 epochs with early stopping (`patience = 10`).

In [ ]:
model = YOLO(T.base_model)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=T.epochs,
    imgsz=T.img_size,
    batch=T.batch_size,
    patience=T.patience,
    seed=cfg.seed,
    project=str(RUNS_DIR),
    name=T.run_name,
    exist_ok=True,
)

## 2 · Keep the best checkpoint

In [ ]:
best = RUNS_DIR / T.run_name / "weights" / "best.pt"
P.weights.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, P.weights)
print(f"best weights -> {P.weights}")

ShowImage(filename=str(RUNS_DIR / T.run_name / "results.png"))   # training curves

## 3 · Validation metrics (Table I)
Expected for **Pathology Tissue**: P 0.911 · R 0.928 · mAP@0.5 **0.949** · mAP@0.5:0.95 0.781.

In [ ]:
detector = YOLO(P.weights)
metrics = detector.val(data=str(DATA_YAML), imgsz=T.img_size, batch=T.batch_size,
                       split="val", plots=False)

table = pd.DataFrame(
    [(detector.names[c], *metrics.box.class_result(i))
     for i, c in enumerate(metrics.box.ap_class_index)],
    columns=["class", "precision", "recall", "mAP@0.5", "mAP@0.5:0.95"],
).round(3)
table